Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits as follows:  • Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account", "general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g. if d%3 gives "account", write a question like “how do I update my registered mobile number”).  • # Example roll number ...23 -> digits 2, 3 • # digit 2 -> category[2 % 3] = general • # digit 3 -> category[3 % 3] = billing

In [1]:
import pandas as pd

ROLL_NUMBER = "2310990123"  # <-- replace with your actual roll number

faqs = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
     {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
      {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
       {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
    {"question": "how can i reset my payment pin", "answer":"Go to My Account > Passwords > Login Settings > Change Pin.","keywords":"Change login pin", "category": "account" },
    {"question": "where can i access printable reciepts for my payment", "answer": "Go to Billings > Paid Bills > Reciepts.", "keywords": "payment reciepts download", "category": "billing"},
    ]

# Turn the list of dicts into an actual DataFrame -- this was missing before,
# which is why every later cell that used "df" failed with NameError.
df = pd.DataFrame(faqs)
df["num_keywords"] = df["keywords"].apply(lambda kw: len(kw.split()))
df


,question,answer,keywords,category,num_keywords
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing,4
1,how to reset password,Go to Settings > Reset Password.,password reset login,account,3
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general,4
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,4
4,how can i reset my payment pin,Go to My Account > Passwords > Login Settings ...,Change login pin,account,3
5,where can i access printable reciepts for my p...,Go to Billings > Paid Bills > Reciepts.,payment reciepts download,billing,3


Q2: Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence


In [2]:
def score(query, df):
  words=set(query.lower().split())
  scores=[]
  for _, row in df.iterrows():
    keyword_set=set(row["keywords"].lower().split())
    question_set=set(row["question"].lower().split())

    overlap=len(words&keyword_set)+len(words&question_set)   # fixed: was query_words (undefined)
    scores.append(overlap)
  result = df.copy()
  result["score"]=scores
  result=result[result["score"]>0].sort_values("score", ascending=False)
  return result.reset_index(drop=True)

score("fee", df)


,question,answer,keywords,category,num_keywords,score
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing,4,2
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,4,2


Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.


In [3]:
def same_category(category_name, dataframe):
    return dataframe.loc[dataframe["category"] == category_name, ["question", "category"]]

sample_category = df.loc[4, "category"]
print(f"Category chosen: '{sample_category}'")
print(same_category(sample_category, df))


Category chosen: 'account'
                         question category
1           how to reset password  account
4  how can i reset my payment pin  account


Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.


In [4]:
index=0
print(f"Selected entry to update: '{df.loc[index, 'question']}'")

new_keyword = input("Enter a new keyword to add to this entry: ").strip()
df.loc[index, "keywords"] = df.loc[index, "keywords"] + " " + new_keyword

df["num_keywords"] = df["keywords"].apply(lambda kw: len(kw.split()))

csv_filename = f"{ROLL_NUMBER}_faq_data.csv"
df.to_csv(csv_filename, index=False)
print(f"Updated DataFrame saved to '{csv_filename}'")
print(df)


Selected entry to update: 'what is the annual fee'
Enter a new keyword to add to this entry: password
Updated DataFrame saved to '2310990123_faq_data.csv'
                                            question  \
0                             what is the annual fee   
1                              how to reset password   
2                        what are your working hours   
3                              how can i pay the fee   
4                     how can i reset my payment pin   
5  where can i access printable reciepts for my p...   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Go to My Account > Passwords > Login Settings ...   
5            Go to Billings > Paid Bills > Reciepts.   

                         keywords category  num_keywords  


Q5: Using groupby, print how many FAQ entries you have per category.


In [5]:
category_summary = df.groupby("category").agg(
    num_entries=("question", "count"),
    min_keywords=("num_keywords", "min"),
    max_keywords=("num_keywords", "max"),
    avg_keywords=("num_keywords", "mean"),
)
print(category_summary)
print()

          num_entries  min_keywords  max_keywords  avg_keywords
category                                                       
account             2             3             3           3.0
billing             3             3             5           4.0
general             1             4             4           4.0



Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't


In [6]:
def score_query_with_tie_handling(query, dataframe):
    scored = score(query, dataframe)   # fixed: was score_query (never defined)

    if scored.empty:
        print(f"No matches found for query: '{query}'")
        return scored

    top_score = scored["score"].max()
    top_matches = scored[scored["score"] == top_score]

    if len(top_matches) > 1:
        print(f"Query: '{query}' -> {len(top_matches)} entries TIE for the top score ({top_score}):")
    else:
        print(f"Query: '{query}' -> Best match (score {top_score}):")

    print(top_matches[["question", "answer", "category", "score"]])
    return top_matches

score_query_with_tie_handling("fee", df)
print()

score_query_with_tie_handling("reset password login", df)


Query: 'fee' -> 2 entries TIE for the top score (2):
                 question                                      answer  \
0  what is the annual fee                   The annual fee is Rs 500.   
1   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category  score  
0  billing      2  
1  billing      2  

Query: 'reset password login' -> Best match (score 5):
                question                            answer category  score
0  how to reset password  Go to Settings > Reset Password.  account      5


,question,answer,keywords,category,num_keywords,score
0,how to reset password,Go to Settings > Reset Password.,password reset login,account,3,5
